# Week 02 Lab — Complexity Analysis Practice

## Objectives
- Measure time complexity experimentally and compare it with theoretical analysis.
- Understand how algorithm choice affects performance by comparing O(n²) vs O(n) solutions.
- Visualize execution time growth for different complexity classes.

## Prerequisites
- Python 3.10+ installed
- `matplotlib` package: `pip install matplotlib`

---
## A-1: Write a Timing Utility (10 min)

Before we can compare algorithms, we need a reliable way to measure execution time.
We use `time.perf_counter()` for high-resolution timing, and run each function multiple
times to get a stable average.

In [ ]:
import time
import random


def measure_time(func, *args, repeat=3):
    """Run func(*args) multiple times and return average time in seconds."""
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        result = func(*args)
        end = time.perf_counter()
        times.append(end - start)
    avg = sum(times) / len(times)
    return avg, result

### Test the timing utility

Let's measure a simple `sum_list` function across different input sizes to see how the time scales.

In [ ]:
def sum_list(arr):
    """Sum all elements in arr using a loop. O(n)."""
    total = 0
    for x in arr:
        total += x
    return total


print(f"{'N':>10} | {'Time (sec)':>12}")
print("-" * 28)
for n in [1_000, 10_000, 100_000, 1_000_000]:
    data = [random.randint(1, 100) for _ in range(n)]
    elapsed, _ = measure_time(sum_list, data)
    print(f"{n:>10,} | {elapsed:>12.6f}")

**Question**: As N grows by 10x, how does the execution time change? Does this match the expected O(n) behavior?

---
## A-2: Finding Duplicates — O(n²) to O(n) Optimization (15 min)

**Problem**: Given an array of integers, determine whether it contains any duplicate elements.

We will implement two solutions:
1. **Brute-force O(n²)**: Compare all pairs using nested for loops
2. **Hash set O(n)**: Use a `set` for constant-time lookups

In [ ]:
def has_duplicate_bruteforce(arr):
    """O(n^2): Check all pairs."""
    n = len(arr)
    for i in range(n):
        for j in range(i + 1, n):
            if arr[i] == arr[j]:
                return True
    return False


def has_duplicate_hashset(arr):
    """O(n): Use a set."""
    seen = set()
    for x in arr:
        if x in seen:
            return True
        seen.add(x)
    return False

### Performance Comparison

Let's measure both approaches at different input sizes. We use arrays with **no duplicates**
(worst case for both algorithms, since they must check all elements).

In [ ]:
print(f"{'N':>10} | {'O(n\u00b2)':>12} | {'O(n)':>12} | {'Speedup':>8}")
print("-" * 50)

for n in [100, 1_000, 10_000, 50_000]:
    data = list(range(n))  # No duplicates (worst case for both)
    random.shuffle(data)

    t1, _ = measure_time(has_duplicate_bruteforce, data)
    t2, _ = measure_time(has_duplicate_hashset, data)
    speedup = t1 / t2 if t2 > 0 else float('inf')

    print(f"{n:>10,} | {t1:>12.6f} | {t2:>12.6f} | {speedup:>7.1f}x")

### Discussion

1. **How does the speedup change as N increases?** Why does this happen?
2. **What is the space complexity trade-off?** The hash set approach uses extra memory. How much?
3. **When might the brute-force approach be acceptable?** Think about very small N.

---
## Exercise: Implement Your Own Duplicate Finder

Instead of just detecting duplicates, **return the first duplicate value found**.
Implement both O(n²) and O(n) versions.

In [ ]:
def find_first_duplicate_bruteforce(arr):
    """O(n^2): Return the first duplicate value, or None if no duplicates."""
    n = len(arr)
    # TODO: Use nested loops to find the first element that appears more than once.
    # Return the value (not the index) of the first duplicate found.
    pass


def find_first_duplicate_hashset(arr):
    """O(n): Return the first duplicate value, or None if no duplicates."""
    # TODO: Use a set to track seen elements.
    # Return the value as soon as you see it a second time.
    pass


# Test
test1 = [3, 1, 4, 1, 5, 9, 2, 6]
test2 = [1, 2, 3, 4, 5]

print(f"Test 1: {test1}")
print(f"  Brute-force: {find_first_duplicate_bruteforce(test1)}")
print(f"  Hash set:    {find_first_duplicate_hashset(test1)}")
print(f"Test 2: {test2}")
print(f"  Brute-force: {find_first_duplicate_bruteforce(test2)}")
print(f"  Hash set:    {find_first_duplicate_hashset(test2)}")

---
## A-3: Execution Time Graphs (10 min)

Let's measure and plot execution times for four complexity classes:
- **O(1)**: Constant time (access first element)
- **O(n)**: Linear time (sum all elements)
- **O(n log n)**: Linearithmic time (sort the array)
- **O(n²)**: Quadratic time (nested loop)

In [ ]:
def o_1(arr):
    """O(1): Access first element."""
    return arr[0] if arr else None


def o_n(arr):
    """O(n): Sum all elements."""
    total = 0
    for x in arr:
        total += x
    return total


def o_n_log_n(arr):
    """O(n log n): Sort the array."""
    return sorted(arr)


def o_n_squared(arr):
    """O(n^2): Nested loop."""
    n = len(arr)
    count = 0
    for i in range(n):
        for j in range(n):
            count += 1
    return count

In [ ]:
def measure(func, arr, repeat=3):
    """Measure average execution time."""
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        func(arr)
        end = time.perf_counter()
        times.append(end - start)
    return sum(times) / len(times)


sizes = [100, 500, 1000, 2000, 5000, 10000]
results = {"O(1)": [], "O(n)": [], "O(n log n)": [], "O(n\u00b2)": []}

for n in sizes:
    data = [random.randint(1, 1000) for _ in range(n)]
    results["O(1)"].append(measure(o_1, data))
    results["O(n)"].append(measure(o_n, data))
    results["O(n log n)"].append(measure(o_n_log_n, data))
    results["O(n\u00b2)"].append(measure(o_n_squared, data))
    print(f"N={n:>6}: O(1)={results['O(1)'][-1]:.6f}, O(n)={results['O(n)'][-1]:.6f}, "
          f"O(n log n)={results['O(n log n)'][-1]:.6f}, O(n\u00b2)={results['O(n\u00b2)'][-1]:.6f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
for label, times in results.items():
    plt.plot(sizes, times, marker='o', label=label)
plt.xlabel("Input Size (N)")
plt.ylabel("Time (seconds)")
plt.title("Execution Time by Complexity Class")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

### Discussion

1. **Which complexity class grows the fastest?** At what N does O(n²) become noticeably slower?
2. **Why does O(1) appear as a flat line?** What does this tell us about constant-time operations?
3. **Can you tell the difference between O(n) and O(n log n) visually?** Why or why not?

---
## Type B — Web Code Analysis (Discussion)

### B-1: Product Search API Comparison

The `examples/web_search_api/` folder contains a Flask app with two search endpoints:
- `GET /search/linear?q=product_name` — Linear search O(n)
- `GET /search/binary?q=product_name` — Binary search O(log n)

Run the Flask app separately in your terminal:
```bash
cd examples/web_search_api
pip install flask
python app.py
```

Then compare response times with N=100, 10,000, and 1,000,000 products.

**Discussion Question**: How does the difference between linear and binary search change as the data grows larger? At what scale does it become critical to use binary search?

---
## Summary

In this lab, you:
- Built a **timing utility** using `time.perf_counter()` for measuring execution time
- Compared **O(n²) brute-force** vs **O(n) hash set** approaches for finding duplicates
- Measured and plotted execution times for **O(1), O(n), O(n log n), and O(n²)** complexity classes
- Observed how algorithm choice dramatically affects performance as input size grows

**Key takeaway**: Choosing the right algorithm (and data structure) can mean the difference between milliseconds and minutes at scale.

**Next week**: Sorting algorithms — from O(n²) basics to O(n log n) advanced sorts.